# 01 · Warm start: baseline and retrieval variants (Chapter 7)

Reproduces **Table 7.8** (and the full-graph row of Table 7.7).

| Step | Thesis source |
|---|---|
| graph, features, split | `THIS_IS_THE_ONE_I_RUN` cells 6–12 |
| five training runs | same notebook, cells 24–33 (`run_variant`) |
| 5-draw strict evaluation | `rerun_hidden384` cells 2 and 5 |

The training logs print sklearn's **default** macro-F1 (what early stopping used). Report only the strict numbers from the evaluation section.
Known issues kept as in the thesis: hidden width = number of classes (212), label-based rare routing.

In [ ]:
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"   # PLACEHOLDER: thesis inputs (read only)
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"   # PLACEHOLDER: new outputs
TEST_MODE = True     # True: small test subgraph from notebook 00, few epochs
                     # False: full thesis graph (needs about 64 GB of RAM)
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [ ]:
import os, sys
# The package is installed in the venv by setup/setup_local.sh (editable install).
# Run this notebook with the "ddi (5090)" kernel.
try:
    import ddi
except ImportError:
    raise SystemExit('ddi is not importable: run "bash setup/setup_local.sh" and select '
                     'the "ddi (5090)" kernel.')
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(ddi.__file__), "..", ".."))
print("python:", sys.executable)
print("ddi from:", os.path.dirname(ddi.__file__))

In [ ]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT)
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

In [ ]:
from ddi import graph as G
from ddi.splits import filter_and_split_graph, save_split, check_split

if TEST_MODE:
    pair = os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin")
    assert os.path.exists(pair), "Run notebook 00 first to build the test subgraph."
    subgraph, line_subgraph = G.load_graph_pair(pair)
else:
    G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
    subgraph, line_subgraph = G.load_graph_pair(paths.graph_bin)   # 640M edges: high-RAM only
print(subgraph); print(line_subgraph)

line_subgraph = G.attach_endpoint_features(subgraph, line_subgraph)
g_lg_filt = filter_and_split_graph(line_subgraph, label_key="label")
del line_subgraph; gc.collect()

# write the split once, then check every later run against it
RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "full"
split_path = os.path.join(paths.warm_dir, f"split_{RUN_TAG}.pt")
if os.path.exists(split_path):
    check_split(g_lg_filt, split_path)
else:
    save_split(g_lg_filt, split_path)
NUM_CLASSES = int(g_lg_filt.ndata["label"].max().item() + 1)
print("NUM_CLASSES:", NUM_CLASSES)

## Train
Set `TRAIN = False` to skip training and only evaluate existing checkpoints.

In [ ]:
from ddi import train

TRAIN = True
CKPT_DIR = os.path.join(paths.warm_dir, f"checkpoints_{RUN_TAG}")
KNN_DIR  = os.path.join(paths.warm_dir, f"knn_{RUN_TAG}")
os.makedirs(KNN_DIR, exist_ok=True)

run_args = dict(train.COMMON_RUN_ARGS)
if TEST_MODE:
    run_args["num_epochs"] = 2

results = {}
if TRAIN:
    for name, kw in train.WARM_START_VARIANTS.items():
        k = kw.get("proto_k")
        cache = os.path.join(KNN_DIR, f"knn_cache_k{k}.pt") if k else None
        _, m = train.run_variant(g_lg_filt, name, checkpoint_dir=CKPT_DIR, knn_cache_path=cache,
                                 device=device, **run_args, **kw)
        results[name] = m
        gc.collect(); torch.cuda.empty_cache()
    json.dump(results, open(os.path.join(paths.warm_dir, f"train_log_metrics_{RUN_TAG}.json"), "w"),
              indent=2, default=str)

## Evaluate (Table 7.8)
Five inference draws per checkpoint, strict macro-F1 over event classes 1..211 and rare-class F1 (training count < 15).

`EVAL_THESIS_CHECKPOINTS = True` re-evaluates the original thesis checkpoints instead (full graph only). Their kNN cache for k=5 is read from `DATA_ROOT`; k=3 is rebuilt.

In [ ]:
from ddi import evaluate

EVAL_THESIS_CHECKPOINTS = False
if EVAL_THESIS_CHECKPOINTS:
    assert not TEST_MODE, "thesis checkpoints only fit the full graph"
    eval_dir = paths.thesis_warm_ckpt_dir
    import shutil
    if os.path.exists(paths.thesis_knn_cache_k5) and not os.path.exists(os.path.join(KNN_DIR, "knn_cache_k5.pt")):
        shutil.copy(paths.thesis_knn_cache_k5, os.path.join(KNN_DIR, "knn_cache_k5.pt"))
else:
    eval_dir = CKPT_DIR

table_7_8 = evaluate.evaluate_checkpoint_dir(
    g_lg_filt, eval_dir, n_draws=5, device=device, knn_cache_dir=KNN_DIR,
    out_json=os.path.join(paths.warm_dir, f"table_7_8_{RUN_TAG}{'_thesis' if EVAL_THESIS_CHECKPOINTS else ''}.json"))

In [ ]:
print("Thesis Table 7.8 (concat k=5): macro-F1 0.814 +/- 0.004, rare-F1 0.705; "
      "baseline (single draw) 0.729 / 0.369")